Data Validation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install duckdb

import os
import re
import duckdb
import pandas as pd

BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"
START_YEAR, END_YEAR = 1992, 2023

# expected column names (adjust if needed)
EXPECTED_COLS = [
    "exporter", "importer", "commoditycode", "value_exporter", "value_importer"
]

def parquet_files_in_range(base, start, end):
    files = []
    for y in range(start, end + 1):
        fp = os.path.join(base, f"H0_{y}.parquet")
        files.append((y, fp, os.path.exists(fp)))
    return files

def schema_for_file(con, path):
    # DESCRIBE reads schema without scanning full file
    return con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()

def sample_for_file(con, path, n=5):
    return con.execute(f"SELECT * FROM read_parquet('{path}') LIMIT {n}").df()

con = duckdb.connect()

rows = []
missing_files = []
missing_cols = []
type_mismatches = []

baseline_schema = None

for year, fp, exists in parquet_files_in_range(BASE, START_YEAR, END_YEAR):
    if not exists:
        missing_files.append(year)
        rows.append({"year": year, "status": "missing_file"})
        continue

    try:
        sch = schema_for_file(con, fp)  # columns: column_name, column_type, null, key, default, extra
        cols = sch["column_name"].tolist()
        colset = set(cols)

        # Check expected columns present
        miss = [c for c in EXPECTED_COLS if c not in colset]
        if miss:
            missing_cols.append((year, miss))

        # Establish baseline schema from first available file
        if baseline_schema is None:
            baseline_schema = sch[["column_name", "column_type"]].copy()
            baseline_schema = baseline_schema.set_index("column_name")["column_type"].to_dict()

        # Compare column types for columns that are in baseline
        current_types = sch.set_index("column_name")["column_type"].to_dict()
        mism = []
        for c, t0 in baseline_schema.items():
            if c in current_types and current_types[c] != t0:
                mism.append((c, t0, current_types[c]))
        if mism:
            type_mismatches.append((year, mism))

        # Quick data sanity from small sample (optional)
        smp = sample_for_file(con, fp, n=5)
        rows.append({
            "year": year,
            "status": "ok",
            "n_cols": len(cols),
            "has_expected_cols": (len(miss) == 0),
        })

    except Exception as e:
        rows.append({"year": year, "status": "error", "error": str(e)})

con.close()

report = pd.DataFrame(rows).sort_values("year")

print("Missing files:", missing_files)
print("\nYears missing expected columns (year, missing_cols):")
print(missing_cols[:20], "..." if len(missing_cols) > 20 else "")

print("\nYears with type mismatches vs baseline (year, [(col, baseline, current), ...]):")
print(type_mismatches[:10], "..." if len(type_mismatches) > 10 else "")

display(report)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Missing files: []

Years missing expected columns (year, missing_cols):
[] 

Years with type mismatches vs baseline (year, [(col, baseline, current), ...]):
[] 


,year,status,n_cols,has_expected_cols
0,1992,ok,7,True
1,1993,ok,7,True
2,1994,ok,7,True
3,1995,ok,7,True
4,1996,ok,7,True
5,1997,ok,7,True
6,1998,ok,7,True
7,1999,ok,7,True
8,2000,ok,7,True
9,2001,ok,7,True


Moran I and Confidence interval

In [ ]:
# ============================================================
# Moran's I by year (1992–2023) with:
#  - Permutation interval (null envelope) from m.sim quantiles
#  - Normal-approx interval using VI_norm (assumes normality)
#
# Output CSV includes I + both interval types for exports/imports.
# Stand-alone: reads OD_Matrix.csv + H0_YYYY.parquet files.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip -q install duckdb libpysal esda scipy

import os
import numpy as np
import pandas as pd
import duckdb

from esda.moran import Moran
from libpysal.weights import W as psW
from scipy.stats import norm

# ----------------------------
# Config
# ----------------------------
BASE = "/content/drive/My Drive/96 Colab Notebooks/Growth Lab"
OD_PATH = os.path.join(BASE, "OD_Matrix.csv")

START_YEAR, END_YEAR = 1992, 2023
PERMUTATIONS = 999
ALPHA = 0.05  # 95%

EXPORTER_COL = "exporter"
IMPORTER_COL = "importer"
VALUE_EXPORT_COL = "value_exporter"
VALUE_IMPORT_COL = "value_importer"

OUT_CSV = os.path.join(BASE, "moran_trade_1992_2023_perm_and_norm_intervals.csv")

# ----------------------------
# Load OD once -> W_full once (inverse-distance, row-standardized)
# ----------------------------
od_long = pd.read_csv(OD_PATH)
od_long["origin"] = od_long["origin"].astype(str).str.upper()
od_long["destination"] = od_long["destination"].astype(str).str.upper()

D_full = od_long.pivot(index="origin", columns="destination", values="distance_km")
all_codes = sorted(set(D_full.index) | set(D_full.columns))
D_full = D_full.reindex(index=all_codes, columns=all_codes)

np.fill_diagonal(D_full.values, 0.0)

D = D_full.to_numpy(dtype=float)
np.fill_diagonal(D, np.nan)
W_full = 1.0 / D
W_full = W_full / np.nansum(W_full, axis=1, keepdims=True)

code_to_ix = {c: i for i, c in enumerate(all_codes)}

# ----------------------------
# Helpers
# ----------------------------
def log10_1p(x):
    return np.log10(np.asarray(x, dtype=float) + 1.0)

def subset_weights_dense(codes):
    idx = np.array([code_to_ix[c] for c in codes], dtype=int)
    W = W_full[np.ix_(idx, idx)].copy()
    np.fill_diagonal(W, 0.0)

    rs = W.sum(axis=1, keepdims=True)
    rs[rs == 0] = np.nan
    W = W / rs

    neighbors, weights = {}, {}
    for i, c in enumerate(codes):
        js = np.where(np.isfinite(W[i]) & (W[i] > 0))[0]
        neighbors[c] = [codes[j] for j in js]
        weights[c] = [float(W[i, j]) for j in js]

    w = psW(neighbors, weights)
    w.transform = "R"
    return w

def agg_year_trade(parquet_path: str):
    con = duckdb.connect()
    exports = con.execute(f"""
        SELECT UPPER({EXPORTER_COL}) AS ISO_A3, SUM({VALUE_EXPORT_COL}) AS exports
        FROM read_parquet('{parquet_path}')
        GROUP BY 1
    """).df()
    imports = con.execute(f"""
        SELECT UPPER({IMPORTER_COL}) AS ISO_A3, SUM({VALUE_IMPORT_COL}) AS imports
        FROM read_parquet('{parquet_path}')
        GROUP BY 1
    """).df()
    con.close()
    return exports, imports

def moran_with_intervals(codes, values, permutations=999, alpha=0.05):
    values = np.asarray(values, dtype=float)
    codes = [str(c).upper() for c in codes]

    # drop non-finite
    mask = np.isfinite(values)
    codes = [c for c, ok in zip(codes, mask) if ok]
    values = values[mask]

    # keep only codes present in OD universe
    kept_codes, kept_vals = [], []
    for c, v in zip(codes, values):
        if c in code_to_ix:
            kept_codes.append(c)
            kept_vals.append(v)
    values = np.asarray(kept_vals, dtype=float)
    codes = kept_codes

    if len(values) < 5 or np.std(values) == 0:
        return {
            "n": int(len(values)),
            "I": np.nan,
            "perm_low": np.nan, "perm_high": np.nan,
            "norm_low": np.nan, "norm_high": np.nan
        }

    # stable ordering
    order = np.argsort(codes)
    codes = [codes[i] for i in order]
    values = values[order]

    w = subset_weights_dense(codes)
    m = Moran(values, w, permutations=permutations)

    # Permutation interval from null distribution of I (m.sim)
    perm_low, perm_high = np.percentile(
        m.sim, [100*alpha/2, 100*(1-alpha/2)]
    ).astype(float)

    # Normal-approx interval for I: I ± z * sqrt(VI_norm)
    zcrit = norm.ppf(1 - alpha/2)
    if np.isfinite(m.VI_norm) and m.VI_norm >= 0:
        se = float(np.sqrt(m.VI_norm))
        norm_low = float(m.I - zcrit * se)
        norm_high = float(m.I + zcrit * se)
    else:
        norm_low, norm_high = (np.nan, np.nan)

    return {
        "n": int(len(values)),
        "I": float(m.I),
        "perm_low": float(perm_low),
        "perm_high": float(perm_high),
        "norm_low": float(norm_low),
        "norm_high": float(norm_high),
    }

# ----------------------------
# Run for all years
# ----------------------------
rows = []
for y in range(START_YEAR, END_YEAR + 1):
    fp = os.path.join(BASE, f"H0_{y}.parquet")
    if not os.path.exists(fp):
        rows.append({"year": y, "status": "missing_file"})
        continue

    exp_df, imp_df = agg_year_trade(fp)

    exp_df["log_exports"] = log10_1p(exp_df["exports"].to_numpy())
    imp_df["log_imports"] = log10_1p(imp_df["imports"].to_numpy())

    exp_res = moran_with_intervals(
        exp_df["ISO_A3"].tolist(), exp_df["log_exports"].to_numpy(),
        permutations=PERMUTATIONS, alpha=ALPHA
    )
    imp_res = moran_with_intervals(
        imp_df["ISO_A3"].tolist(), imp_df["log_imports"].to_numpy(),
        permutations=PERMUTATIONS, alpha=ALPHA
    )

    rows.append({
        "year": y,
        "status": "ok",

        "n_exports": exp_res["n"],
        "I_exports": exp_res["I"],
        "I_exports_ci95_perm_low": exp_res["perm_low"],
        "I_exports_ci95_perm_high": exp_res["perm_high"],
        "I_exports_ci95_norm_low": exp_res["norm_low"],
        "I_exports_ci95_norm_high": exp_res["norm_high"],

        "n_imports": imp_res["n"],
        "I_imports": imp_res["I"],
        "I_imports_ci95_perm_low": imp_res["perm_low"],
        "I_imports_ci95_perm_high": imp_res["perm_high"],
        "I_imports_ci95_norm_low": imp_res["norm_low"],
        "I_imports_ci95_norm_high": imp_res["norm_high"],
    })

out = pd.DataFrame(rows).sort_values("year")
out.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
out.head(10)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/drive/My Drive/96 Colab Notebooks/Growth Lab/moran_trade_1992_2023_perm_and_norm_intervals.csv


,year,status,n_exports,I_exports,I_exports_ci95_perm_low,I_exports_ci95_perm_high,I_exports_ci95_norm_low,I_exports_ci95_norm_high,n_imports,I_imports,I_imports_ci95_perm_low,I_imports_ci95_perm_high,I_imports_ci95_norm_low,I_imports_ci95_norm_high
0,1992,ok,157,0.089688,-0.027216,0.019377,0.066722,0.112655,157,0.089089,-0.027706,0.019359,0.066123,0.112056
1,1993,ok,161,0.066809,-0.028206,0.017454,0.044474,0.089143,161,0.062948,-0.025768,0.018823,0.040613,0.085283
2,1994,ok,161,0.074583,-0.026010,0.017203,0.052248,0.096918,161,0.069276,-0.025289,0.020969,0.046941,0.091611
3,1995,ok,161,0.052555,-0.026895,0.018450,0.030220,0.074890,161,0.045687,-0.025659,0.020395,0.023352,0.068022
4,1996,ok,161,0.057242,-0.025990,0.020271,0.034908,0.079577,161,0.055946,-0.025489,0.019500,0.033611,0.078281
5,1997,ok,161,0.036460,-0.026248,0.021266,0.014125,0.058794,161,0.064442,-0.025067,0.018459,0.042107,0.086777
6,1998,ok,161,0.065233,-0.026591,0.019676,0.042898,0.087568,161,0.065569,-0.024546,0.018806,0.043234,0.087903
7,1999,ok,162,0.059806,-0.025394,0.020156,0.037524,0.082088,162,0.056931,-0.025192,0.017471,0.034649,0.079214
8,2000,ok,167,0.031781,-0.025638,0.021428,0.009478,0.054083,167,0.032915,-0.024377,0.019398,0.010613,0.055217
9,2001,ok,167,0.039477,-0.024482,0.017711,0.017174,0.061779,167,0.037677,-0.025258,0.019669,0.015374,0.059979
